# C1: Inside the decision model (Strands hidden states)

Strands 2B (Qwen3.5-2B-Base torso, 24 layers, pointer head) reads `<state>` text `</state>` → `<question>` header + question text → `<options>` lines → `<answer>` causally. For exactly the frozen feature-creation requests (per text column and joint, identical deduplication) one label-free forward saves, in float16, the hidden state at the last token of the text, the last token of the question and the `<answer>` token, at layer 12 and layer 24 (the layer the pointer head reads). C1a: 3 positions × 2 layers × {per text column (PCA32 per column), joint (PCA64)} = 12 arms; C1b: the `<answer>` layer-24 vector together with Strands' own probabilities, per column and joint = 2 arms. PCA is fitted on outer-training rows only. Common protocol (Group B): seven fixed predictors with library defaults, the no-text baseline, 18 datasets, the same five outer folds; fixed headline metrics (binary ROC-AUC, multiclass macro one-vs-rest ROC-AUC, regression R²); datasets are the unit, with equal-dataset means, dataset-bootstrap intervals and win/tie/loss counts. No frozen encoder enters this comparison.

Tables come from `output_JEVPFN/exp09_representations/results/group_c/c1_representations.zip`.


In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import group_c, study_export
from src.visualize import figures, group_b as gbfig, group_c as gcfig, style
style.apply()
NOTEBOOK = "exp09_representations"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. Scope, provenance and the export read by this notebook")
display_json("Notebook execution identity", provenance())
frames, receipt = group_c.read("C1")
print(study_export.describe(receipt))
record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, "Export identity")
WAITING = "Waiting for export: no measurements are shown."

Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\exp09_representations\results\group_c\c1_representations.zip is absent. No measurements are shown.


## Design and extraction contract

In [2]:
report_section("2. Design and extraction contract")
record(group_c.design_counts(), "Group C arms, configurations and C2 fits (plan, not results)")
if not frames:
    print(WAITING)
else:
    display_table("Hidden-state stores per dataset: coverage, reproduction of Strands' probabilities by the released head on the saved float16 vectors, and size", frames["extraction"], preview_rows=18)
    record(frames["extraction"], "Extraction stores")
    display_table("Planned versus complete five-fold arms per predictor", frames["coverage"], preview_rows=14)
    record(frames["coverage"], "Coverage")

Waiting for export: no measurements are shown.


## Gains of the hidden-state arms

In [3]:
report_section("3. Gains of the hidden-state arms over the no-text baseline")
if not frames:
    print(WAITING)
else:
    for view in ("per_column", "joint"):
        gbfig.render(save, gcfig.c1_gains(frames["summary"], view=view, name=f"c1_gains_{view}"))
    display_table("Equal-dataset mean gain per arm and predictor", frames["summary"], preview_rows=20)
    record(frames["summary"], "Gain summary")

Waiting for export: no measurements are shown.


## Comparisons

In [4]:
report_section("4. Comparisons: vector versus probabilities, positions and layers")
if not frames:
    print(WAITING)
else:
    summary = frames["contrast_summary"]
    names = sorted(set(summary.representation)) if not summary.empty else []
    groups = {
        "c1_vector_vs_probabilities": ("<answer> vector against and with Strands' own probabilities", ("answer_last",)),
        "c1_position_path": ("End of text, end of question and <answer> at each layer", ("question_end_vs", "answer_vs")),
        "c1_middle_vs_last": ("Layer 24 against layer 12 at each position", ("last_vs_middle",)),
    }
    for name, (what, prefixes) in groups.items():
        rows = [n for n in names if n.startswith(prefixes)]
        gbfig.render(save, gcfig.contrasts(summary, rows, name=name, what=what))
    display_table("Fold-paired contrasts (A minus B), equal-dataset summary", summary, preview_rows=20)
    record(summary, "Contrast summary")

Waiting for export: no measurements are shown.


## Complete report

In [5]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified export was read; no inference, model training or live VSC changes were performed.")

exp09_representations

1. Scope, provenance and the export read by this notebook


Notebook execution identity
{
  "code_sha256": "6fefc8d71fb8d4ede1d1459074ab16e1630d80996c3504f09c91cc61f2f6d4c7",
  "git_head": "afff4cf5f464ef78b0e6d086b12e595a153fe72f",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Export identity
{
  "status": "awaiting_export",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\exp09_representations\\results\\group_c\\c1_representations.zip",
  "created_utc": null,
  "origin": null,
  "sha256": null,
  "bytes": null,
  "schema": nul